# Capstone 1 — RAG Pipeline (Plan A) auf Google Colab

**Was dieses Notebook macht:**
1. GPU prüfen + Google Drive einbinden
2. Pakete installieren
3. Index bauen: Chunking → S-PubMedBERT Embedding → FAISS + BM25
4. Retrieval testen

**Vor dem Start:**
- Laufzeit auf **T4 GPU** stellen: *Laufzeit → Laufzeittyp ändern → T4 GPU*
- `clean_articles_enriched.pkl` auf Google Drive hochladen
- Pfad in **Zelle 3** anpassen

## Schritt 1 — GPU prüfen + Google Drive einbinden

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print(f'GPU verfuegbar: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNUNG: Kein GPU gefunden!')
    print('Bitte: Laufzeit -> Laufzeittyp aendern -> T4 GPU')

## Schritt 2 — Pakete installieren

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu tqdm -q
print('Alle Pakete installiert.')

## Schritt 3 — Konfiguration

**WICHTIG:** Pfad zur pkl-Datei anpassen!

In [ ]:
from pathlib import Path

# ============================================================
# HIER ANPASSEN: Wo liegt die pkl-Datei in deinem Google Drive?
# Beispiel: /content/drive/MyDrive/capstone/clean_articles_enriched.pkl
PKL_PATH = Path('/content/drive/MyDrive/capstone/clean_articles_enriched.pkl')
# ============================================================

# Index wird hier gespeichert (auf Drive -> bleibt erhalten)
INDEX_DIR   = PKL_PATH.parent / 'faiss_v2'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

FAISS_INDEX = INDEX_DIR / 'pubmedbert.index'
FAISS_META  = INDEX_DIR / 'pubmedbert_meta.pkl'
BM25_PATH   = INDEX_DIR / 'bm25.pkl'

# Modell + Chunking
EMBED_MODEL         = 'pritamdeka/S-PubMedBert-MS-MARCO'
TEXT_FIELDS         = ['title', 'abstract', 'introduction']
CHUNK_MAX_CHARS     = 1600
CHUNK_OVERLAP_CHARS = 200
BATCH_SIZE          = 128   # GPU: groessere Batches moeglich (war 32 auf CPU)

# Retrieval
TOP_K_FAISS  = 20
TOP_K_BM25   = 20
TOP_K_OUTPUT = 15
MIN_SCORE    = 0.45
RRF_K        = 60

print(f'PKL  : {PKL_PATH}')
print(f'Index: {INDEX_DIR}')
print(f'PKL vorhanden: {PKL_PATH.exists()}')

## Schritt 4 — Hilfsfunktionen (Chunker + Embedder)

In [ ]:
import numpy as np
import pickle
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Chunker (identisch mit python/lib/chunker.py) ─────────────────────────────
_SEPARATORS = ['\n\n', '\n', '. ', ' ']

def chunk_text(text, max_chars=1600, overlap_chars=200):
    text = text.strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            cut = -1
            for sep in _SEPARATORS:
                pos = text.rfind(sep, start + overlap_chars, end)
                if pos != -1:
                    cut = pos + len(sep)
                    break
            if cut != -1:
                end = cut
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap_chars, start + 1)
    return chunks

def build_text(row, fields, sep=' [SEP] '):
    parts = [str(row.get(f) or '').strip() for f in fields]
    return sep.join(p for p in parts if p)

# ── Embedder ──────────────────────────────────────────────────────────────────
def load_embedder(model_name, batch_size=128):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Lade Modell: {model_name} auf {device} ...')
    model = SentenceTransformer(model_name, device=device)
    return model, device, batch_size

def embed(model, texts, batch_size=128):
    vecs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    return vecs

print('Hilfsfunktionen geladen.')

## Schritt 5 — Index bauen

Falls der Index bereits auf Drive existiert, wird er einfach geladen (schnell).
Falls nicht, wird er jetzt gebaut (~5-10 Min auf T4 GPU).

In [ ]:
import pandas as pd

# ── Index laden falls vorhanden ────────────────────────────────────────────────
if FAISS_INDEX.exists() and FAISS_META.exists() and BM25_PATH.exists():
    print('Index gefunden — wird geladen ...')
    faiss_index = faiss.read_index(str(FAISS_INDEX))
    with open(FAISS_META, 'rb') as f:
        meta = pickle.load(f)
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)
    model, device, _ = load_embedder(EMBED_MODEL, BATCH_SIZE)
    print(f'Index geladen: {faiss_index.ntotal} Vektoren | {len(meta)} Chunks')

# ── Index neu bauen ────────────────────────────────────────────────────────────
else:
    print('Kein Index gefunden — wird jetzt gebaut ...')

    # 1. Artikel laden
    print(f'Lade {PKL_PATH.name} ...')
    with open(PKL_PATH, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, pd.DataFrame):
        records = data.to_dict('records')
    elif isinstance(data, dict):
        records = list(data.values())
    else:
        records = data
    print(f'{len(records)} Artikel geladen.')

    # 2. Chunking
    print('Chunking ...')
    all_chunks, meta = [], []
    for row in tqdm(records):
        text = build_text(row, TEXT_FIELDS)
        chunks = chunk_text(text, CHUNK_MAX_CHARS, CHUNK_OVERLAP_CHARS)
        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            meta.append({
                'pmcid':             str(row.get('pmcid', row.get('pmc_id', ''))),
                'title':             str(row.get('title', '')),
                'cluster_name':      str(row.get('cluster_name', '')),
                'kwd_lst_cl':        row.get('kwd_lst_cl', []),
                'tfidf_candidates':  row.get('tfidf_candidates', []),
                'chunk_idx':         i,
                'chunk_text':        chunk,
            })
    print(f'Total Chunks: {len(all_chunks)} (Ø {len(all_chunks)/len(records):.1f} pro Artikel)')

    # 3. Embedding (GPU!)
    model, device, _ = load_embedder(EMBED_MODEL, BATCH_SIZE)
    print(f'Embedding {len(all_chunks)} Chunks auf {device} ...')
    embeddings = embed(model, all_chunks, BATCH_SIZE)
    print(f'Embeddings: {embeddings.shape}')

    # 4. FAISS Index
    print('Baue FAISS IndexFlatIP ...')
    dim = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(embeddings)
    faiss.write_index(faiss_index, str(FAISS_INDEX))
    print(f'FAISS gespeichert: {FAISS_INDEX}')

    # 5. BM25
    print('Baue BM25 ...')
    tokenized = [c.lower().split() for c in all_chunks]
    bm25 = BM25Okapi(tokenized)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)

    # 6. Meta
    with open(FAISS_META, 'wb') as f:
        pickle.dump(meta, f)

    print(f'Index komplett gespeichert in: {INDEX_DIR}')
    print(f'FAISS: {faiss_index.ntotal} Vektoren | BM25: {len(bm25.corpus_size)} Chunks')

## Schritt 6 — Retrieval testen

Hier kannst du beliebige Suchanfragen eingeben und schauen welche Artikel gefunden werden.

In [ ]:
# Retrieval-Funktionen (identisch mit python/lib/retriever.py)

def faiss_search(index, query_vec, top_k, min_score):
    scores, indices = index.search(query_vec.reshape(1, -1), top_k)
    return [(int(i), float(s)) for i, s in zip(indices[0], scores[0])
            if i >= 0 and s >= min_score]

def bm25_search(bm25, query, top_k):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx if scores[i] > 0]

def rrf_merge(faiss_hits, bm25_hits, k=60):
    scores = {}
    for rank, (idx, _) in enumerate(faiss_hits):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    for rank, (idx, _) in enumerate(bm25_hits):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def dedup_articles(hits, meta, top_k):
    seen = {}
    for idx, score in hits:
        entry = meta[idx]
        pmcid = entry['pmcid']
        if pmcid not in seen or score > seen[pmcid]['retrieval_score']:
            seen[pmcid] = {**entry, 'retrieval_score': score}
    ranked = sorted(seen.values(), key=lambda x: x['retrieval_score'], reverse=True)
    return ranked[:top_k]

def retrieve(query_text):
    # Embed query
    q_vec = model.encode([query_text], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    # Search
    f_hits = faiss_search(faiss_index, q_vec, TOP_K_FAISS, MIN_SCORE)
    b_hits = bm25_search(bm25, query_text, TOP_K_BM25)
    merged = rrf_merge(f_hits, b_hits, RRF_K)
    return dedup_articles(merged, meta, TOP_K_OUTPUT)

print('Retrieval-Funktionen bereit.')

In [ ]:
# ── HIER ANFRAGE EINGEBEN ──────────────────────────────────────────────────────
QUERY = 'CRISPR gene editing cancer treatment'
# ──────────────────────────────────────────────────────────────────────────────

results = retrieve(QUERY)

print(f'\nTop {len(results)} Artikel fuer: "{QUERY}"')
print('=' * 70)
for i, r in enumerate(results, 1):
    print(f'{i:2}. [{r["retrieval_score"]:.4f}] {r["title"]}')
    print(f'     Cluster : {r["cluster_name"]}')
    kwd = r.get('kwd_lst_cl', [])
    if kwd:
        print(f'     Keywords: {", ".join(str(k) for k in kwd[:5])}')
    print()

## (Optional) Schritt 7 — Visualisierung: t-SNE der Embeddings

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import random

SAMPLE_N = 800

# Zufaellige Stichprobe
total = faiss_index.ntotal
sample_idx = sorted(random.sample(range(total), min(SAMPLE_N, total)))

# Vektoren aus FAISS holen
vecs = faiss.rev_swig_ptr(faiss_index.get_xb(), total * faiss_index.d)
vecs = np.array(vecs).reshape(total, faiss_index.d)[sample_idx]
labels = [meta[i]['cluster_name'] for i in sample_idx]

# t-SNE
print('Berechne t-SNE (ca. 1-2 Min) ...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
coords = tsne.fit_transform(vecs)

# Plot
unique_labels = sorted(set(labels))
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
label_to_color = dict(zip(unique_labels, colors))

fig, ax = plt.subplots(figsize=(14, 9))
for lbl in unique_labels:
    mask = [l == lbl for l in labels]
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=[label_to_color[lbl]], label=lbl, s=8, alpha=0.6)

ax.set_title(f't-SNE der Chunk-Embeddings (n={SAMPLE_N})', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, markerscale=2)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/capstone/embedding_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: embedding_tsne.png')